# Guided Solver Config Builder

Configure your model inputs and write `solver_params.json` to disk.
Run this notebook before `guided_bayesian_inference.ipynb`.

In [1]:
from solver_config_builder import build_solver_params_config, write_solver_params_json_file

In [2]:
# =======================
# FILE PATHS
# =======================
# Use a reaction folder containing one or more reaction JSON files.
reactions_source = "./Test_Model/Reactions"

solver_file_dir = "./Test_Model"
solver_params_file = "solver_params_test.json"

# Set save directory and output file names
savedir = "./Test_Model/Results"
prior_samples_file = "Test_Model_prior_samples_pm.nc"
posterior_samples_file = "Test_Model_posterior_samples_pm.nc"
trace_plot_file = "Test_Model_trace_plot.png"

# =============================
# FREE PARAMETERS
# =============================
# Required fields: rxn_name, param_name, distribution, lower, upper
# Optional fields: mass (default: 0.95), fixed_stat
free_parameter_prior_inputs = [
    {"rxn_name": "binding_forward", "param_name": "k_f", "distribution": "LogNormal", "lower": 1.0e-3, "upper": 1.0, "mass": 0.95, "fixed_stat": ("mean", 1.0e-1)},
    {"rxn_name": "binding_reverse", "param_name": "k_r", "distribution": "LogNormal", "lower": 1.0e-3, "upper": 1.0, "mass": 0.95},
    {"rxn_name": "catalysis", "param_name": "k_cat", "distribution": "LogNormal", "lower": 1.0e-2, "upper": 10.0, "mass": 0.95},
    {"rxn_name": "side_product", "param_name": "k_side", "distribution": "LogNormal", "lower": 5.0e-4, "upper": 2.0, "mass": 0.95},
]

# ========================
# DATASETS
# ========================
# This fixture set is designed to exercise all supported dataset types and noise models.
dataset_inputs = [
    {
        "name": "ts_rel_mean",
        "dataset_type": "timeseries",
        "data_file": "./Test_Model/Data/ts_relative_mean.csv",
        "observable": "product_fraction",
        "time_column": "time",
        "column_mapping": {"product_fraction": "product_fraction"},
        "noise_model": "relative_mean",
        "noise_params": {"frac": 0.05},
    },
    {
        "name": "ts_rel_pointwise",
        "dataset_type": "timeseries",
        "data_file": "./Test_Model/Data/ts_relative_pointwise.csv",
        "observable": "product_concentration",
        "time_column": "time",
        "column_mapping": {"product_concentration": "product_concentration"},
        "init_cond_columns": {"S": "S_init"},
        "noise_model": "relative_pointwise",
        "noise_params": {"frac": 0.08, "floor": 0.002},
    },
    {
        "name": "ts_rel_plus_floor",
        "dataset_type": "timeseries",
        "data_file": "./Test_Model/Data/ts_relative_plus_floor.csv",
        "observable": "total_substrate_pool",
        "time_column": "time",
        "column_mapping": {"total_substrate_pool": "total_substrate_pool"},
        "noise_model": "relative_plus_floor",
        "noise_params": {"frac": 0.03, "floor": 0.001},
    },
    {
        "name": "ts_absolute",
        "dataset_type": "timeseries",
        "data_file": "./Test_Model/Data/ts_absolute_total_pool.csv",
        "observable": "total_product_pool",
        "time_column": "time",
        "column_mapping": {"total_product_pool": "total_product_pool"},
        "noise_model": "absolute",
        "noise_params": {"value": 0.025},
    },
    {
        "name": "ts_column",
        "dataset_type": "timeseries",
        "data_file": "./Test_Model/Data/ts_column_sigma.csv",
        "observable": "product_fraction",
        "time_column": "time",
        "column_mapping": {"product_fraction": "product_fraction"},
        "noise_model": "column",
        "sigma_column_mapping": {"product_fraction": "product_sigma"},
    },
    {
        "name": "replicate_groupwise",
        "dataset_type": "replicate_timeseries",
        "data_file": "./Test_Model/Data/replicate_groupwise.csv",
        "observable": "product_fraction",
        "time_column": "time",
        "column_mapping": {"product_fraction": "product_fraction"},
        "init_cond_columns": {"S": "S_init", "E": "E_init"},
        "noise_model": "groupwise",
        "noise_params": {"group_column": "replicate_id", "statistic": "std", "min_sigma": 1.0e-4},
    },
    {
        "name": "endpoint_absolute",
        "dataset_type": "endpoint",
        "data_file": "./Test_Model/Data/endpoint_absolute.csv",
        "observable": "final_product_concentration",
        "time_values": [4.0],
        "column_mapping": {"final_product_concentration": "final_product_concentration"},
        "init_cond_columns": {"S": "S_init", "E": "E_init"},
        "noise_model": "absolute",
        "noise_params": {"value": 0.02},
    },
    {
        "name": "endpoint_rel_mean",
        "dataset_type": "endpoint",
        "data_file": "./Test_Model/Data/endpoint_relative_mean.csv",
        "observable": "final_product_concentration",
        "time_values": [4.0],
        "column_mapping": {"final_product_concentration": "final_product_concentration"},
        "init_cond_columns": {"S": "S_init"},
        "noise_model": "relative_mean",
        "noise_params": {"frac": 0.07},
    },
    {
        "name": "rate_column",
        "dataset_type": "rate",
        "data_file": "./Test_Model/Data/rate_column_sigma.csv",
        "observable": "product_formation_rate",
        "time_column": "time",
        "column_mapping": {"product_formation_rate": "product_formation_rate"},
        "noise_model": "column",
        "sigma_column_mapping": {"product_formation_rate": "rate_sigma"},
    },
    {
        "name": "rate_groupwise",
        "dataset_type": "rate",
        "data_file": "./Test_Model/Data/rate_groupwise.csv",
        "observable": "product_formation_rate",
        "time_column": "time",
        "column_mapping": {"product_formation_rate": "product_formation_rate"},
        "noise_model": "groupwise",
        "noise_params": {"group_column": "batch", "statistic": "sem", "min_sigma": 1.0e-4},
    },
]

# =====================================
# SAMPLING + SOLVER SETTINGS
# =====================================
prior_sampling_settings = {"draws": 100, "random_seed": 2026}
posterior_sampling_settings = {
    "draws": 100,
    "tune": 100,
    "chains": 4,
    "random_seed": 2026,
    "target_accept": 0.95,
    "nuts_sampler": "nutpie"
}

ode_solver_settings = {
    "solver_name": "Kvaerno5",
    "dt0": 1.0e-12,
    "max_steps": 10000000,
    "stepsize_controller": "PIDController",
}

ode_stepsize_controller_settings = {
    "rtol": 1.0e-6,
    "atol": 1.0e-8,
    "pcoeff": 0.3,
    "icoeff": 0.4,
    "dcoeff": 0.0,
}

# =================================
# MODEL-SPECIFIC SETTINGS
# =================================
calculation_module_path = "./Test_Model/test_calculations.py"
initial_conditions = {"S": 1.0, "E": 0.1, "ES": 0.0, "P": 0.0, "Q": 0.0}

In [3]:
solver_params = build_solver_params_config(
    free_parameter_prior_inputs=free_parameter_prior_inputs,
    dataset_inputs=dataset_inputs,
    prior_sampling_settings=prior_sampling_settings,
    posterior_sampling_settings=posterior_sampling_settings,
    ode_solver_settings=ode_solver_settings,
    ode_stepsize_controller_settings=ode_stepsize_controller_settings,
    calculation_module_path=calculation_module_path,
    initial_conditions=initial_conditions,
)

written_solver_params_path = write_solver_params_json_file(
    solver_params_config=solver_params,
    file_directory=solver_file_dir,
    filename=solver_params_file,
)

print(f"Wrote solver config: {written_solver_params_path}")
print(f"Reaction source: {reactions_source}")
print(f"Save directory: {savedir}")
print(
    f"Configured free params: {[param['param_name'] for param in solver_params['free_kinetic_params']]}"
)
print(
    f"Configured datasets: {[dataset['name'] for dataset in solver_params['datasets']]}"
)

Wrote solver config: /Users/annettethompson/Library/CloudStorage/OneDrive-SharedLibraries-UCB-O365/Jerome Michael Fox - Annie Thompson/Git Repositories/Bayesian Kinetic Model/Restructured Framework/Test_Model/solver_params_test.json
Reaction source: ./Test_Model/Reactions
Save directory: ./Test_Model/Results
Configured free params: ['k_f', 'k_r', 'k_cat', 'k_side']
Configured datasets: ['ts_rel_mean', 'ts_rel_pointwise', 'ts_rel_plus_floor', 'ts_absolute', 'ts_column', 'replicate_groupwise', 'endpoint_absolute', 'endpoint_rel_mean', 'rate_column', 'rate_groupwise']
